In [1]:
import pandas as pd

# --- EDIT THIS PATH ---
PATH = "p1_run_ma.csv"

df = pd.read_csv(PATH)

# Normalize column names just in case
df.columns = [c.strip() for c in df.columns]

required = ["TokenID", "FullText", "t_surf", "t_o"]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}\nFound: {list(df.columns)}")

# Basic sanity checks
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nMissing values in required columns:")
print(df[required].isna().sum())

# Drop rows with missing required fields (should be 0 if everything is clean)
df_clean = df.dropna(subset=required).copy()
print("\nRows after dropping missing required fields:", len(df_clean))

# Ensure types
df_clean["t_o"] = df_clean["t_o"].astype(int)

df_clean.head()

Rows: 594
Columns: 65

Missing values in required columns:
TokenID     0
FullText    0
t_surf      0
t_o         0
dtype: int64

Rows after dropping missing required fields: 594


,TokenID,t_o,FullText,Target,t_surf,DtExp,CS,Reason,DtRes,CS.1,...,CS.16,Reason.16,AktSemel,CS.17,Reason.17,eng_gram,cmn_gram,deu_gram,jpn_gram,metadata
0,40001018,1,The birth genesis : origin beginning of Jesus ...,had been promised,had been promised,0,High,"""Had been promised"" describes a change of soc...",1,High,...,--,"OR ""Had been promised"" describes a state resul...",0,High,"""Had been promised"" implies a state resulting...","past, participle",le,"prat, indikativ, sein, partizip",n_a,NaN
1,40001024,1,"When Joseph woke up , he did what the angel of...",had commanded,had commanded,1,High,"""Had commanded"" is a verb of communication, f...",0,High,...,--,OR,0,High,"The verb ""command"" implies a directive that i...","participle, past",suo_de,"prat, indikativ, haben, partizip",ta,NaN
2,40003002,1,"» Repent , for the kingdom of heaven has come ...",has come near,has come near,0,High,"The verb phrase ""has come near"" describes a c...",1,High,...,--,OR,0,High,"""Has come near"" implies a change of state (th...","past, participle",le,"prasens, indikativ, sein, partizip",ta,NaN
3,40003017,1,Then a voice from heaven said : » This is my S...,have approved,have approved,0,--,OR,1,High,...,High,"""Have approved"" describes an instantaneous tr...",0,High,"""Have approved"" implies a change of state (ap...","present, participle",suo_de,"prasens, indikativ, haben, partizip",n_a,NaN
4,40004012,1,Jesus heard that John had been arrested . So J...,had been arrested,had been arrested,0,High,The state of being arrested is relevant to Je...,1,High,...,High,"""Had been arrested"" describes an instantaneou...",0,High,"The phrase ""had been arrested"" implies a chan...","past, participle",compound,"prat, indikativ, sein, werden, partizip",ta,NaN


In [2]:
def find_nth_span(text, sub, n):
    """
    Find start/end character span of the nth occurrence of sub in text.
    n is 1-based.
    Returns (start, end) or (None, None) if not found.
    """
    start = 0
    count = 0
    while True:
        idx = text.find(sub, start)
        if idx == -1:
            return None, None
        count += 1
        if count == n:
            return idx, idx + len(sub)
        start = idx + len(sub)

df_clean[["char_start", "char_end"]] = df_clean.apply(
    lambda row: pd.Series(
        find_nth_span(row["FullText"], row["t_surf"], row["t_o"])
    ),
    axis=1
)

In [3]:
# Quick sanity check
missing_spans = df_clean[df_clean["char_start"].isna()]
print("Rows with missing spans:", len(missing_spans))

missing_spans[["TokenID", "t_surf", "t_o", "FullText"]].head()

Rows with missing spans: 0


,TokenID,t_surf,t_o,FullText


In [ ]:
# Report missing spans
#for _, row in missing_spans.iterrows():
#    print("TokenID:", row["TokenID"])
#    print("t_surf:", repr(row["t_surf"]))
#    print("t_o:", row["t_o"])
#    print("FullText:", row["FullText"])
#    print("-" * 80)

In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-uncased"  # solid default for English

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()  # inference mode

# If you have a GPU available, this will use it; otherwise CPU is fine.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Loaded:", MODEL_NAME, "| device:", device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loaded: bert-base-uncased | device: cpu


In [5]:
# Smoke-test tokenization + span alignment on one row
# Retrieve a representative example
row = df_clean.iloc[0]

text = row["FullText"]
start = row["char_start"]
end = row["char_end"]

print("TokenID:", row["TokenID"])
print("t_surf:", row["t_surf"])
print("Span text:", repr(text[start:end]))
print("FullText:", text)

TokenID: 40001018
t_surf: had been promised
Span text: 'had been promised'
FullText: The birth genesis : origin beginning of Jesus Christ was as follows : His mother Mary had been promised to Joseph in marriage . But before they were married , Mary realized that she was pregnant by Holy Spirit .


In [6]:
# Smoke-test tokenization + span alignment on one row
# Tokenize with offsets
encoded = tokenizer(
    text,
    return_offsets_mapping=True,
    return_tensors="pt"
)

offsets = encoded["offset_mapping"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

# Show tokens that overlap the target span
for tok, (s, e) in zip(tokens, offsets):
    if e > start and s < end:
        print(tok, (s, e))

had (86, 89)
been (90, 94)
promised (95, 103)


In [7]:
# Extract one token-span embedding (and [CLS]) for that same row

"""
- run the verse through BERT
- grab the final-layer hidden states
- compute:
 - span_vec = mean of the token vectors overlapping your char span
 - cls_vec = the [CLS] vector for the whole verse
- print shapes so we know everything’s sane
"""

import numpy as np

def bert_span_and_cls(text, char_start, char_end, tokenizer, model, device):
    # Tokenize with offsets
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        return_tensors="pt",
        truncation=True,  # safety
        max_length=512
    )
    offsets = enc["offset_mapping"][0].tolist()

    # Move tensors to device (but keep offsets on CPU)
    enc = {k: v.to(device) for k, v in enc.items() if k != "offset_mapping"}

    with torch.no_grad():
        out = model(**enc)  # last_hidden_state: (1, seq_len, hidden)
        hs = out.last_hidden_state[0]  # (seq_len, hidden)

    # Build mask for tokens overlapping the char span
    mask = []
    for (s, e) in offsets:
        overlaps = (e > char_start) and (s < char_end)
        mask.append(overlaps)

    idx = torch.tensor([i for i, m in enumerate(mask) if m], device=device, dtype=torch.long)
    if idx.numel() == 0:
        raise ValueError("No tokens overlap the target span; check offsets/span.")

    span_vec = hs.index_select(0, idx).mean(dim=0)  # (hidden,)
    cls_vec = hs[0]  # [CLS] is position 0 in BERT-style models

    return span_vec.detach().cpu().numpy(), cls_vec.detach().cpu().numpy(), idx.detach().cpu().tolist()

# Use the same row you tested
text = row["FullText"]
start = int(row["char_start"])
end = int(row["char_end"])

span_vec, cls_vec, token_idx = bert_span_and_cls(text, start, end, tokenizer, model, device)

print("Token indices used:", token_idx)
print("span_vec shape:", span_vec.shape)
print("cls_vec shape:", cls_vec.shape)

Token indices used: [17, 18, 19]
span_vec shape: (768,)
cls_vec shape: (768,)


In [8]:
# Batch over all rows and build two embedding matrices

"""
- loop through df_clean
- compute X_span and X_cls
- track any rows that get truncated past 512 tokens (rare for verses, but we log it)
- produce numpy arrays you can save next
"""

from tqdm.auto import tqdm

def embed_dataframe(df, tokenizer, model, device):
    X_span = []
    X_cls = []
    bad_rows = []
    truncated = 0

    for i, row in tqdm(df.iterrows(), total=len(df)):
        text = row["FullText"]
        cs = int(row["char_start"])
        ce = int(row["char_end"])

        # Tokenize (keep offsets)
        enc = tokenizer(
            text,
            return_offsets_mapping=True,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        offsets = enc["offset_mapping"][0].tolist()

        # Detect truncation (fast tokenizer sets overflow by cutting offsets)
        # If any offset is (0,0) after content tokens, or if last offset ends before end of text.
        # Simple heuristic:
        if offsets and offsets[-1][1] < len(text):
            truncated += 1

        enc = {k: v.to(device) for k, v in enc.items() if k != "offset_mapping"}

        try:
            with torch.no_grad():
                out = model(**enc)
                hs = out.last_hidden_state[0]  # (seq_len, hidden)

            idx = [j for j, (s, e) in enumerate(offsets) if (e > cs and s < ce)]
            if len(idx) == 0:
                raise ValueError("No overlap tokens")

            idx_t = torch.tensor(idx, device=device, dtype=torch.long)
            span_vec = hs.index_select(0, idx_t).mean(dim=0)
            cls_vec = hs[0]

            X_span.append(span_vec.detach().cpu().numpy())
            X_cls.append(cls_vec.detach().cpu().numpy())

        except Exception as e:
            bad_rows.append((i, row["TokenID"], str(e)))
            # Append NaNs to keep alignment (optional); here we skip to keep arrays clean
            # If you prefer alignment preservation, tell me and we’ll adjust.
            continue

    return np.vstack(X_span), np.vstack(X_cls), bad_rows, truncated

X_span, X_cls, bad_rows, truncated = embed_dataframe(df_clean, tokenizer, model, device)

print("X_span:", X_span.shape)
print("X_cls :", X_cls.shape)
print("Bad rows:", len(bad_rows))
print("Truncated texts:", truncated)
if bad_rows:
    print("First few bad rows:", bad_rows[:5])

  0%|          | 0/594 [00:00<?, ?it/s]

X_span: (594, 768)
X_cls : (594, 768)
Bad rows: 0
Truncated texts: 594


In [12]:
# Save the embedding matrices
import numpy as np

np.save("bert_span_embeddings.npy", X_span)
np.save("bert_cls_embeddings.npy", X_cls)

# Save minimal metadata needed for joining later
meta = df_clean[["TokenID"]].reset_index(drop=True)
meta.to_csv("bert_metadata.csv", index=False)